[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/26.%20The%20Gate%20Automation%20%26%20Damage%20Detection%20Problem/P26-Tier-5_executed.ipynb)

---


# 26. The Gate Automation and Damage Detection Problem

## Tier 5: The Integrated Digital Twin

### Goal

This notebook builds a small, synthetic system-of-systems simulation for an automated gate facility. It represents sensors, component degradation, predictive-model outputs, operational scenarios, and coordination among gates, security, energy, and traffic systems. The purpose is to show how these elements can be connected in one simulation workflow—not to validate a real facility or train a machine-learning model.

### Inputs and outputs

The inputs are the facility configuration, model parameters, and a simulation duration. The simulation generates synthetic sensor readings, updates physics-model states, produces illustrative predictions and alerts, records subsystem metrics, and evaluates three what-if scenarios. The final analyses summarize those generated records and plot selected time series.

### Scope and modeling assumptions

- The facility contains 12 gates. The sensor layout below creates 432 synthetic sensors; this is a modeling choice for this notebook, not a reproduction of the source example's reported 600-plus sensors.
- Sensor values are generated by simple equations with random noise. They are not readings from an operational IoT network.
- The wear and electrical-degradation equations are illustrative, simplified physics-inspired models. Their parameters and outputs require calibration before any engineering use.
- The predictive models are rule-based simulations that produce example outputs. No machine-learning model is trained, and the stated model accuracies are configuration values rather than measured validation results.
- Scenario outcomes are calculated from explicit proportional rules. They are not causal estimates or experimentally validated benefits.
- Security and energy events use random draws. Since no random seed is set, event counts and associated results can differ between runs.
- The simulation does not model all real operational constraints, sensor failures, communication delays, uncertainty in model parameters, or the full dynamics of a port facility.

### Method and execution order

The simulation follows a discrete-time sequence: advance the simulation clock; generate sensor data; update component degradation; run predictive models; coordinate subsystem responses; and record subsystem performance. Scenario analysis is then run separately, followed by summaries and visualizations. This order makes each simulation step's inputs and outputs explicit.

For a time step of length $\Delta t$, a state variable $x_t$ is updated by a model-specific transition rule:

$$x_{t+1} = f(x_t, u_t, \Delta t, \varepsilon_t),$$

where $x_t$ is the current simulated state, $u_t$ represents modeled operating conditions, and $\varepsilon_t$ represents random variation when present. In the code, `simulation_step` controls the sequence, while the sensor, physics, prediction, and coordination methods implement the individual parts. The model records states at each step; it does not solve an optimization problem or establish that a policy is optimal.

### What to look for

Interpret the outputs as consequences of the stated synthetic rules. Check whether the simulation records data for the configured duration, whether component state variables move in plausible directions, and whether each scenario's reported outcomes follow its calculation. An alert or a favorable scenario outcome is a model-generated event, not evidence that a real failure was predicted or a real saving was achieved.

The source example describes a major logistics hub and reports specific outcomes, including an advance failure warning, speed-related benefits, demand-response revenue, and reduced lockdown time. Those reports provide context for the application area; this notebook does not reproduce or independently verify them.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
import random
from typing import Any, Dict, List, Optional
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class IoTSensor:
    """Represent a sensor and its recent synthetic readings."""

    sensor_id: str
    gate_id: int
    sensor_type: str
    location: str
    sampling_rate: float
    accuracy: float
    current_value: float = 0.0
    historical_data: List[float] = field(default_factory=list)
    health_status: str = "normal"


@dataclass
class PhysicsModel:
    """Represent a simplified component-degradation model."""

    component_id: str
    model_type: str
    parameters: Dict[str, float]
    current_state: Dict[str, float]
    degradation_rate: float = 0.0
    predicted_rul: float = 1000.0


@dataclass
class PredictiveModel:
    """Store configuration and the latest simulated model output."""

    model_id: str
    model_type: str
    accuracy: float
    confidence_threshold: float = 0.8
    last_prediction: Dict[str, Any] = field(default_factory=dict)


@dataclass
class SubSystem:
    """Represent a facility subsystem and its performance metrics."""

    system_id: str
    system_type: str
    status: str
    performance_metrics: Dict[str, float]
    integration_points: List[str]


@dataclass
class Scenario:
    """Store the inputs and reference outcomes for a what-if scenario."""

    scenario_id: str
    name: str
    parameters: Dict[str, Any]
    expected_outcomes: Dict[str, float]
    risk_assessment: float


@dataclass
class DigitalTwinInstance:
    """Group the simulated facility components and its configuration."""

    facility_id: str
    num_gates: int
    sensors: List[IoTSensor]
    physics_models: List[PhysicsModel]
    predictive_models: List[PredictiveModel]
    subsystems: List[SubSystem]
    scenarios: List[Scenario]
    current_time: datetime
    simulation_speed: float = 1.0

In [ ]:
def create_digital_twin_facility() -> DigitalTwinInstance:
    """Build the synthetic 12-gate facility used in this notebook."""

    facility_id = "LOGISTICS_HUB_001"
    num_gates = 12
    sensors = []
    sensor_types = ["vibration", "thermal", "current", "environmental"]
    locations = ["motor", "gearbox", "sensor_array", "control_system"]
    sensor_id = 0

    # This fixed layout produces 36 sensors per gate, or 432 in total.
    for gate_id in range(1, num_gates + 1):
        for sensor_type in sensor_types:
            for location in locations:
                sensors_per_location = 3 if sensor_type == "vibration" else 2
                for _ in range(sensors_per_location):
                    sensor_id += 1
                    sensor_settings = {
                        "vibration": (1000.0, 0.95),
                        "thermal": (10.0, 0.98),
                        "current": (50.0, 0.99),
                        "environmental": (1.0, 0.90),
                    }
                    sampling_rate, accuracy = sensor_settings[sensor_type]
                    sensors.append(
                        IoTSensor(
                            sensor_id=f"SENSOR_{sensor_id:04d}",
                            gate_id=gate_id,
                            sensor_type=sensor_type,
                            location=location,
                            sampling_rate=sampling_rate,
                            accuracy=accuracy,
                        )
                    )

    physics_models = []
    for gate_id in range(1, num_gates + 1):
        physics_models.extend(
            [
                PhysicsModel(
                    component_id=f"G{gate_id}_MOTOR",
                    model_type="mechanical_wear",
                    parameters={
                        "wear_coefficient": 0.001,
                        "load_factor": 1.0,
                        "speed_factor": 1.0,
                        "temperature_factor": 1.0,
                    },
                    current_state={
                        "wear_level": 0.0,
                        "operating_hours": 0.0,
                        "temperature": 25.0,
                        "vibration_level": 0.1,
                    },
                ),
                PhysicsModel(
                    component_id=f"G{gate_id}_CONTROL",
                    model_type="electrical_degradation",
                    parameters={
                        "voltage_stress_factor": 0.0005,
                        "current_stress_factor": 0.0003,
                        "thermal_stress_factor": 0.0002,
                    },
                    current_state={
                        "insulation_resistance": 1000.0,
                        "capacitance": 100.0,
                        "operating_temperature": 25.0,
                    },
                ),
            ]
        )

    predictive_models = [
        PredictiveModel("FAILURE_PREDICTOR_001", "failure_prediction", 0.89, 0.8),
        PredictiveModel("DEMAND_FORECASTER_001", "demand_forecast", 0.85, 0.7),
        PredictiveModel("OPTIMIZER_001", "optimization", 0.82, 0.75),
    ]

    subsystems = [
        SubSystem(
            "GATES_001", "gates", "active",
            {"availability": 0.95, "processing_time": 45.0, "energy_consumption": 50.0},
            ["SECURITY_001", "ENERGY_001", "TRAFFIC_001"],
        ),
        SubSystem(
            "SECURITY_001", "security", "active",
            {"response_time": 2.0, "detection_rate": 0.98, "false_positive_rate": 0.02},
            ["GATES_001", "TRAFFIC_001"],
        ),
        SubSystem(
            "ENERGY_001", "energy", "active",
            {"power_consumption": 600.0, "demand_response_capacity": 200.0, "efficiency": 0.85},
            ["GATES_001"],
        ),
        SubSystem(
            "TRAFFIC_001", "traffic", "active",
            {"throughput": 2000.0, "queue_time": 5.0, "utilization": 0.75},
            ["GATES_001", "SECURITY_001"],
        ),
    ]

    scenarios = [
        Scenario(
            "SPEED_OPTIMIZATION_001",
            "Variable Gate Speed Analysis",
            {"speed_reduction": 0.30, "traffic_density": "high", "duration_hours": 12},
            {"processing_time_increase": 40.0, "motor_life_extension": 23.0, "energy_savings": 15.0},
            0.1,
        ),
        Scenario(
            "MAINTENANCE_OPTIMIZATION_001",
            "Predictive Maintenance Scheduling",
            {"failure_threshold": 0.8, "maintenance_window": 4.0, "cost_avoidance_target": 25000},
            {"downtime_reduction": 80.0, "maintenance_cost_reduction": 30.0, "availability_improvement": 5.0},
            0.2,
        ),
        Scenario(
            "DEMAND_RESPONSE_001",
            "Energy Demand Response",
            {"price_threshold": 0.15, "reduction_percentage": 20.0, "duration_hours": 6},
            {"revenue_generation": 45000.0, "processing_impact": 10.0, "grid_stability_contribution": 5.0},
            0.15,
        ),
    ]

    return DigitalTwinInstance(
        facility_id=facility_id,
        num_gates=num_gates,
        sensors=sensors,
        physics_models=physics_models,
        predictive_models=predictive_models,
        subsystems=subsystems,
        scenarios=scenarios,
        current_time=datetime.now(),
        simulation_speed=1.0,
    )


digital_twin = create_digital_twin_facility()
assert len(digital_twin.sensors) == 432
assert len(digital_twin.physics_models) == 24
print("Synthetic digital twin facility created.")
print(f"Facility: {digital_twin.facility_id}")
print(f"Gates: {digital_twin.num_gates}")
print(f"Sensors: {len(digital_twin.sensors)}")
print(f"Physics models: {len(digital_twin.physics_models)}")
print(f"Predictive models: {len(digital_twin.predictive_models)}")
print(f"Subsystems: {len(digital_twin.subsystems)}")
print(f"Scenarios: {len(digital_twin.scenarios)}")

## Simulation model

A **state** is the information needed to describe the simulated system at one point in time: sensor readings, component condition, predictions, alerts, and subsystem metrics. A **time step** advances that state by a fixed interval. The simulator uses $\Delta t = 0.1$ hours, or six minutes.

For a motor, the wear model adds a nonnegative wear increment at each step:

$$\Delta w_t = k_w L S T \Delta t\, U_t V_t\left(1 + \frac{\theta_t}{100}\right), \qquad w_{t+1}=w_t+\Delta w_t,$$

where $w_t$ is the wear level, $k_w$ is the wear coefficient, $L$, $S$, and $T$ are configured load, speed, and temperature factors, $U_t$ and $V_t$ are time-varying operating factors, and $\theta_t$ is the modeled temperature. The code stores these quantities in `PhysicsModel.current_state` and `PhysicsModel.parameters`. If the wear limit is $w_{\max}=1$, the estimated remaining useful life (RUL), in hours, is

$$\operatorname{RUL}_t = \max\left(0, \frac{w_{\max}-w_t}{\dot{w}_t}\right),$$

where $\dot{w}_t=\Delta w_t/\Delta t$ is the current wear rate. This estimate assumes that the current rate continues unchanged; it is not a validated forecast.

For the electrical model, resistance is reduced by a fractional degradation amount $q_t$ per step. The code estimates the time to a minimum resistance $R_{\min}$ by assuming that the current fractional degradation rate continues. The corresponding continuous-rate estimate is $\ln(R_t/R_{\min})/\dot{q}_t$, where $R_t$ is current resistance and $\dot{q}_t$ is the fractional loss rate per hour.

Each simulation step visits every sensor, physics model, predictive model, and subsystem. With $S$ sensors, $P$ physics models, $M$ predictive models, and $B$ subsystems, the main update work per step grows approximately as $O(S+P+M+B)$. Storing every reading and recorded state over $N$ steps requires memory proportional to the amount of recorded history, approximately $O(N(S+P+B))$ for this implementation. These counts describe this simulation's bookkeeping, not the computational cost of a high-fidelity industrial digital twin.

In [ ]:
class DigitalTwinSimulator:
    """Run the facility's synthetic, discrete-time simulation."""

    def __init__(self, digital_twin: DigitalTwinInstance):
        self.digital_twin = digital_twin
        self.simulation_time = 0.0
        self.time_step = 0.1
        self.sensor_data_history = defaultdict(list)
        self.physics_state_history = defaultdict(list)
        self.predictions_history = defaultdict(list)
        self.system_performance_history = defaultdict(list)
        self.active_alerts = []
        self.alert_history = []
        self.alerted_components = set()
        self.total_failures_predicted = 0
        self.failure_alerts_generated = 0
        self.maintenance_scheduled = 0
        self.cost_savings = 0.0

    def simulate_sensor_data_ingestion(self):
        """Generate and record one synthetic reading per sensor."""
        for sensor in self.digital_twin.sensors:
            if sensor.sensor_type == "vibration":
                base = 0.1
                wear_factor = 1.0 + self.simulation_time * 0.001
                load_factor = 1.0 + 0.2 * np.sin(self.simulation_time * 0.5)
                sensor.current_value = base * wear_factor * load_factor + np.random.normal(0, 0.01)
                sensor.health_status = (
                    "critical" if sensor.current_value > 0.5
                    else "warning" if sensor.current_value > 0.3
                    else "normal"
                )
            elif sensor.sensor_type == "thermal":
                load_rise = 15.0 * (1.0 + 0.3 * np.sin(self.simulation_time * 0.3))
                ambient_variation = 5.0 * np.sin(self.simulation_time * 0.1)
                sensor.current_value = 25.0 + load_rise + ambient_variation + np.random.normal(0, 1.0)
                sensor.health_status = (
                    "critical" if sensor.current_value > 80.0
                    else "warning" if sensor.current_value > 70.0
                    else "normal"
                )
            elif sensor.sensor_type == "current":
                operation_factor = 1.0 + 0.5 * np.sin(self.simulation_time * 0.4)
                efficiency_factor = 1.0 + self.simulation_time * 0.0005
                sensor.current_value = 10.0 * operation_factor * efficiency_factor + np.random.normal(0, 0.5)
                sensor.health_status = (
                    "critical" if sensor.current_value > 20.0
                    else "warning" if sensor.current_value > 15.0
                    else "normal"
                )
            else:
                daily_cycle = 8.0 * np.sin(self.simulation_time * 0.2)
                sensor.current_value = 20.0 + daily_cycle + np.random.normal(0, 2.0)
                sensor.health_status = "normal"

            sensor.historical_data.append(sensor.current_value)
            if len(sensor.historical_data) > 100:
                sensor.historical_data.pop(0)
            self.sensor_data_history[sensor.sensor_id].append(
                {
                    "time": self.simulation_time,
                    "value": sensor.current_value,
                    "status": sensor.health_status,
                }
            )

    def simulate_physics_models(self):
        """Advance mechanical-wear and electrical-degradation states."""
        for model in self.digital_twin.physics_models:
            if model.model_type == "mechanical_wear":
                parameters = model.parameters
                state = model.current_state
                current_load = 1.0 + 0.3 * np.sin(self.simulation_time * 0.5)
                current_speed = 1.0 + 0.2 * np.sin(self.simulation_time * 0.3)
                current_temp = state["temperature"]
                wear_increment = (
                    parameters["wear_coefficient"]
                    * parameters["load_factor"]
                    * parameters["speed_factor"]
                    * parameters["temperature_factor"]
                    * self.time_step
                    * current_load
                    * current_speed
                    * (1.0 + current_temp / 100.0)
                )
                state["wear_level"] += wear_increment
                state["operating_hours"] += self.time_step
                wear_rate = wear_increment / self.time_step
                model.degradation_rate = wear_rate
                model.predicted_rul = (
                    max(0.0, (1.0 - state["wear_level"]) / wear_rate)
                    if wear_rate > 0
                    else 1000.0
                )

            elif model.model_type == "electrical_degradation":
                parameters = model.parameters
                state = model.current_state
                current_voltage = 1.0
                current_current = 1.0 + 0.2 * np.sin(self.simulation_time * 0.4)
                current_temp = state["operating_temperature"]
                voltage_loss = parameters["voltage_stress_factor"] * current_voltage**2
                current_loss = parameters["current_stress_factor"] * current_current**2
                thermal_loss = parameters["thermal_stress_factor"] * (current_temp / 100.0) ** 2
                fractional_loss = (voltage_loss + current_loss + thermal_loss) * self.time_step
                state["insulation_resistance"] *= 1.0 - fractional_loss
                state["capacitance"] *= 1.0 + fractional_loss * 0.1
                loss_rate = fractional_loss / self.time_step
                model.degradation_rate = loss_rate
                resistance = state["insulation_resistance"]
                minimum_resistance = 100.0
                if loss_rate > 0 and resistance > minimum_resistance:
                    model.predicted_rul = np.log(resistance / minimum_resistance) / loss_rate
                else:
                    model.predicted_rul = 0.0 if resistance <= minimum_resistance else 1000.0

            self.physics_state_history[model.component_id].append(
                {
                    "time": self.simulation_time,
                    "wear_level": model.current_state.get("wear_level", 0.0),
                    "rul": model.predicted_rul,
                    "degradation_rate": model.degradation_rate,
                }
            )

    def run_predictive_models(self):
        """Apply the notebook's illustrative prediction rules."""
        for model in self.digital_twin.predictive_models:
            if model.model_type == "failure_prediction":
                critical_components = []
                for physics_model in self.digital_twin.physics_models:
                    rul = physics_model.predicted_rul
                    if rul < 100.0:
                        confidence = model.accuracy * (1.0 - rul / 100.0)
                        if confidence > model.confidence_threshold:
                            critical_components.append(
                                {
                                    "component_id": physics_model.component_id,
                                    "predicted_failure_time": self.simulation_time + rul,
                                    "confidence": confidence,
                                    "rul": rul,
                                }
                            )
                model.last_prediction = {
                    "type": "failure_prediction",
                    "critical_components": critical_components,
                    "timestamp": self.simulation_time,
                    "num_predictions": len(critical_components),
                }
                self.total_failures_predicted += len(critical_components)

                for component in critical_components:
                    component_id = component["component_id"]
                    if component_id not in self.alerted_components:
                        self.generate_alert(
                            "CRITICAL",
                            f"A failure-risk threshold was reached for {component_id}",
                            f"Illustrative estimated RUL: {component['rul']:.1f} hours; "
                            f"rule-based confidence: {component['confidence']:.1%}",
                            component_id,
                        )
                        self.alerted_components.add(component_id)
                        self.failure_alerts_generated += 1
                        self.cost_savings += 25000.0

            elif model.model_type == "demand_forecast":
                seasonal_factor = 1.0 + 0.3 * np.sin(self.simulation_time * 0.1)
                daily_factor = 1.0 + 0.2 * np.sin(self.simulation_time * 0.5)
                predicted_demand = 2000.0 * seasonal_factor * daily_factor + np.random.normal(0, 100.0)
                model.last_prediction = {
                    "type": "demand_forecast",
                    "predicted_demand": predicted_demand,
                    "confidence": model.accuracy,
                    "timestamp": self.simulation_time,
                }

            elif model.model_type == "optimization":
                recommendations = []
                if self.simulation_time > 50.0:
                    recommendations.append(
                        {
                            "type": "speed_optimization",
                            "recommendation": "Reduce gate speed by 30% during low-demand periods",
                            "expected_benefits": {
                                "processing_time_increase": 40.0,
                                "motor_life_extension": 23.0,
                                "energy_savings": 15.0,
                            },
                            "confidence": model.accuracy,
                        }
                    )
                model.last_prediction = {
                    "type": "optimization",
                    "recommendations": recommendations,
                    "timestamp": self.simulation_time,
                }

            self.predictions_history[model.model_id].append(
                {"time": self.simulation_time, "prediction": model.last_prediction.copy()}
            )

    def generate_alert(self, severity: str, message: str, details: str, component_id: Optional[str] = None):
        """Record an alert in both the active list and the full history."""
        alert = {
            "timestamp": self.simulation_time,
            "severity": severity,
            "message": message,
            "details": details,
            "component_id": component_id,
            "status": "active",
        }
        self.active_alerts.append(alert)
        self.alert_history.append(alert.copy())
        if len(self.active_alerts) > 10:
            self.active_alerts.pop(0)

    def coordinate_subsystems(self):
        """Simulate random security and energy events and their responses."""
        security_system = next(
            (system for system in self.digital_twin.subsystems if system.system_type == "security"),
            None,
        )
        if security_system and random.random() < 0.05:
            self.generate_alert(
                "SECURITY",
                "Simulated unauthorized-entry event",
                "A simulated coordination response is applied to gates and traffic.",
                "SECURITY_001",
            )
            gates_system = next(
                (system for system in self.digital_twin.subsystems if system.system_type == "gates"),
                None,
            )
            traffic_system = next(
                (system for system in self.digital_twin.subsystems if system.system_type == "traffic"),
                None,
            )
            if gates_system:
                gates_system.performance_metrics["availability"] *= 0.5
            if traffic_system:
                traffic_system.performance_metrics["throughput"] *= 0.3
                traffic_system.performance_metrics["queue_time"] *= 3.0
            self.cost_savings += 12000.0

        energy_system = next(
            (system for system in self.digital_twin.subsystems if system.system_type == "energy"),
            None,
        )
        if energy_system and random.random() < 0.1:
            self.generate_alert(
                "ENERGY",
                "Simulated high electricity-price event",
                "A simulated demand-response action is applied to the gates.",
                "ENERGY_001",
            )
            gates_system = next(
                (system for system in self.digital_twin.subsystems if system.system_type == "gates"),
                None,
            )
            if gates_system:
                gates_system.performance_metrics["processing_time"] *= 1.2
                gates_system.performance_metrics["energy_consumption"] *= 0.8
            self.cost_savings += 45000.0 / (24.0 * 365.0) * self.time_step

    def run_scenario_analysis(self, scenario_id: str) -> Optional[Dict[str, float]]:
        """Calculate the configured proportional outcome for one scenario."""
        scenario = next(
            (item for item in self.digital_twin.scenarios if item.scenario_id == scenario_id),
            None,
        )
        if scenario is None:
            return None

        print(f"\nScenario analysis: {scenario.name}")
        if scenario.scenario_id == "SPEED_OPTIMIZATION_001":
            speed_reduction = scenario.parameters["speed_reduction"]
            scale = speed_reduction / 0.3
            actual_outcomes = {
                "processing_time_increase": 45.0 * scale,
                "motor_life_extension": 23.0 * scale,
                "energy_savings": 15.0 * scale,
            }
        elif scenario.scenario_id == "MAINTENANCE_OPTIMIZATION_001":
            threshold = scenario.parameters["failure_threshold"]
            actual_outcomes = {
                "downtime_reduction": 80.0 * threshold,
                "maintenance_cost_reduction": 30.0 * threshold,
                "availability_improvement": 5.0 * threshold,
            }
            self.maintenance_scheduled += 1
        elif scenario.scenario_id == "DEMAND_RESPONSE_001":
            reduction = scenario.parameters["reduction_percentage"]
            scale = reduction / 20.0
            actual_outcomes = {
                "revenue_generation": 45000.0 * scale,
                "processing_impact": 10.0 * scale,
                "grid_stability_contribution": 5.0 * scale,
            }
        else:
            return None

        print("Configured reference values and rule-based scenario calculations:")
        for key, expected_value in scenario.expected_outcomes.items():
            actual_value = actual_outcomes[key]
            percent_difference = (
                abs(actual_value - expected_value) / abs(expected_value) * 100.0
                if expected_value != 0
                else 0.0
            )
            print(
                f"  {key}: reference={expected_value:.1f}, "
                f"calculated={actual_value:.1f}, difference={percent_difference:.1f}%"
            )
        return actual_outcomes

    def simulation_step(self):
        """Advance time and execute the simulation components in order."""
        self.simulation_time += self.time_step
        self.simulate_sensor_data_ingestion()
        self.simulate_physics_models()
        self.run_predictive_models()
        self.coordinate_subsystems()

        for subsystem in self.digital_twin.subsystems:
            if subsystem.system_type == "gates":
                degradation_rate = 0.001 * self.time_step
                subsystem.performance_metrics["availability"] *= 1.0 - degradation_rate
                subsystem.performance_metrics["processing_time"] *= 1.0 + degradation_rate
            self.system_performance_history[subsystem.system_id].append(
                {"time": self.simulation_time, "metrics": subsystem.performance_metrics.copy()}
            )

    def run_simulation(self, duration_hours: float = 24.0):
        """Run a whole number of time steps and report summary metrics."""
        if duration_hours <= 0:
            raise ValueError("duration_hours must be positive")
        num_steps = int(round(duration_hours / self.time_step))
        if not np.isclose(num_steps * self.time_step, duration_hours):
            raise ValueError("duration_hours must be a multiple of the time step")

        print(f"Running synthetic simulation for {duration_hours:.1f} hours.")
        print(f"Time step: {self.time_step:.1f} hours; number of steps: {num_steps}")
        for step in range(num_steps):
            self.simulation_step()
            if (step + 1) % 100 == 0:
                print(
                    f"  Step {step + 1}/{num_steps}: time={self.simulation_time:.1f} hours, "
                    f"active alerts={len(self.active_alerts)}"
                )
        self.print_final_statistics()

    def print_final_statistics(self):
        """Report recorded simulation quantities without treating them as validation."""
        data_points = sum(len(data) for data in self.sensor_data_history.values())
        print("\nSimulation summary")
        print(f"Simulated duration: {self.simulation_time:.1f} hours")
        print(f"Recorded sensor readings: {data_points:,}")
        print(f"Component-level prediction records: {self.total_failures_predicted}")
        print(f"Failure-risk alerts generated: {self.failure_alerts_generated}")
        print(f"Maintenance scenarios evaluated: {self.maintenance_scheduled}")
        print(f"Illustrative accumulated value: ${self.cost_savings:,.2f}")
        print(f"Currently active alerts: {len(self.active_alerts)}")

        gates_system = next(
            (system for system in self.digital_twin.subsystems if system.system_type == "gates"),
            None,
        )
        if gates_system and self.system_performance_history[gates_system.system_id]:
            records = self.system_performance_history[gates_system.system_id]
            avg_availability = np.mean([record["metrics"]["availability"] for record in records])
            avg_processing_time = np.mean([record["metrics"]["processing_time"] for record in records])
            print(f"Mean simulated gate availability: {avg_availability:.1%}")
            print(f"Mean simulated gate processing time: {avg_processing_time:.2f} seconds")

## Run the facility simulation and scenarios

A simulation run uses the configured time step repeatedly; with a 24-hour duration and a 0.1-hour step, the loop performs one update for each time interval. Scenario calculations are separate from that time loop: they evaluate configured what-if rules and do not change the simulation's time-series history. Their reference values are included to make the arithmetic inspectable, not as measured outcomes.

In [ ]:
simulator = DigitalTwinSimulator(digital_twin)
simulator.run_simulation(duration_hours=24.0)

scenario_results = {}
print("\nScenario analysis results")
for scenario in digital_twin.scenarios:
    result = simulator.run_scenario_analysis(scenario.scenario_id)
    if result is not None:
        scenario_results[scenario.scenario_id] = result

print("\nScenario summary")
for scenario_id, outcomes in scenario_results.items():
    scenario = next(item for item in digital_twin.scenarios if item.scenario_id == scenario_id)
    print(f"  {scenario.name}: {len(outcomes)} calculated outcome measures")

In [ ]:
def analyze_predictive_performance(simulator: DigitalTwinSimulator) -> Dict[str, float]:
    """Summarize simulated failure-risk outputs and demand forecasts."""
    print("\nPredictive-model output analysis")
    confidence_scores = []
    demand_predictions = []

    failure_predictor = next(
        (model for model in simulator.digital_twin.predictive_models
         if model.model_type == "failure_prediction"),
        None,
    )
    if failure_predictor and failure_predictor.model_id in simulator.predictions_history:
        prediction_history = simulator.predictions_history[failure_predictor.model_id]
        for prediction_data in prediction_history:
            prediction = prediction_data["prediction"]
            for component in prediction.get("critical_components", []):
                confidence_scores.append(component["confidence"])

        print("\nRule-based failure-risk outputs")
        print(f"  Component-level prediction records: {len(confidence_scores)}")
        if confidence_scores:
            print(f"  Mean rule-based confidence: {np.mean(confidence_scores):.3f}")
            print(
                "  Outputs above 80% confidence: "
                f"{sum(score > 0.8 for score in confidence_scores)}"
            )
        else:
            print("  No component met the configured rule for a failure-risk output.")

    demand_forecaster = next(
        (model for model in simulator.digital_twin.predictive_models
         if model.model_type == "demand_forecast"),
        None,
    )
    if demand_forecaster and demand_forecaster.model_id in simulator.predictions_history:
        forecast_history = simulator.predictions_history[demand_forecaster.model_id]
        demand_predictions = [
            item["prediction"]["predicted_demand"] for item in forecast_history
        ]
        print("\nSynthetic demand forecasts")
        print(f"  Forecast records: {len(demand_predictions)}")
        if demand_predictions:
            print(f"  Mean forecast: {np.mean(demand_predictions):.1f} vehicles per hour")
            print(
                f"  Forecast range: {np.min(demand_predictions):.1f} to "
                f"{np.max(demand_predictions):.1f} vehicles per hour"
            )
        print(f"  Configured model accuracy value: {demand_forecaster.accuracy:.1%} (not validated here)")

    return {
        "failure_prediction_records": len(confidence_scores),
        "mean_rule_based_confidence": float(np.mean(confidence_scores)) if confidence_scores else 0.0,
        "demand_forecast_records": len(demand_predictions),
    }


predictive_stats = analyze_predictive_performance(simulator)

In [ ]:
def analyze_system_integration(simulator: DigitalTwinSimulator) -> Dict[str, Any]:
    """Summarize recorded subsystem events, metrics, and illustrative values."""
    print("\nSystem-integration analysis")
    security_alerts = [alert for alert in simulator.alert_history if alert["severity"] == "SECURITY"]
    energy_alerts = [alert for alert in simulator.alert_history if alert["severity"] == "ENERGY"]
    critical_alerts = [alert for alert in simulator.alert_history if alert["severity"] == "CRITICAL"]

    print("\nRecorded alert events")
    print(f"  Security events: {len(security_alerts)}")
    print(f"  Energy events: {len(energy_alerts)}")
    print(f"  Failure-risk alerts: {len(critical_alerts)}")

    print("\nMean recorded subsystem metrics")
    for subsystem in simulator.digital_twin.subsystems:
        records = simulator.system_performance_history.get(subsystem.system_id, [])
        if not records:
            continue
        metric_names = records[0]["metrics"].keys()
        averages = {
            metric: float(np.mean([record["metrics"][metric] for record in records]))
            for metric in metric_names
        }
        print(f"\n  {subsystem.system_type.title()} subsystem:")
        for metric, value in averages.items():
            if metric in {"availability", "detection_rate", "false_positive_rate", "efficiency", "utilization"}:
                print(f"    {metric}: {value:.1%}")
            elif metric in {"processing_time", "response_time", "queue_time"}:
                print(f"    {metric}: {value:.2f} seconds")
            else:
                print(f"    {metric}: {value:.2f}")

    coordination_benefits = {
        "security_coordination": len(security_alerts) * 12000.0,
        "energy_coordination": len(energy_alerts) * 45000.0 / (24.0 * 365.0) * simulator.simulation_time,
        "predictive_maintenance": simulator.failure_alerts_generated * 25000.0,
    }
    total_value = sum(coordination_benefits.values())
    print("\nIllustrative coordination values (not measured savings)")
    for category, value in coordination_benefits.items():
        print(f"  {category.replace('_', ' ').title()}: ${value:,.2f}")
    print(f"  Total: ${total_value:,.2f}")

    return {
        "coordination_benefits": coordination_benefits,
        "total_coordination_value": total_value,
        "security_events": len(security_alerts),
        "energy_events": len(energy_alerts),
    }


integration_stats = analyze_system_integration(simulator)

In [ ]:
def create_digital_twin_visualizations(
    simulator: DigitalTwinSimulator,
    predictive_stats: Dict[str, float],
    integration_stats: Dict[str, Any],
):
    """Plot representative sensor, prediction, coordination, and wear histories."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Synthetic Gate-Facility Digital Twin", fontsize=16, fontweight="bold")

    sensor_ax = axes[0, 0]
    sample_sensors = simulator.digital_twin.sensors[:4]
    for sensor in sample_sensors:
        data = simulator.sensor_data_history.get(sensor.sensor_id, [])[-50:]
        if data:
            times = [item["time"] for item in data]
            values = [item["value"] for item in data]
            sensor_ax.plot(times, values, label=f"{sensor.sensor_type.title()} (Gate {sensor.gate_id})")
    sensor_ax.set_title("Sample synthetic sensor readings")
    sensor_ax.set_xlabel("Simulation time (hours)")
    sensor_ax.set_ylabel("Sensor value (type-specific units)")
    sensor_ax.grid(True, alpha=0.3)
    if sample_sensors:
        sensor_ax.legend()

    prediction_ax = axes[0, 1]
    failure_predictor = next(
        (model for model in simulator.digital_twin.predictive_models
         if model.model_type == "failure_prediction"),
        None,
    )
    confidence_scores = []
    prediction_times = []
    if failure_predictor:
        for record in simulator.predictions_history.get(failure_predictor.model_id, []):
            for component in record["prediction"].get("critical_components", []):
                confidence_scores.append(component["confidence"])
                prediction_times.append(record["time"])
    if confidence_scores:
        prediction_ax.scatter(prediction_times, confidence_scores, alpha=0.6, s=20, label="Rule-based output")
        prediction_ax.axhline(
            y=failure_predictor.confidence_threshold,
            color="red",
            linestyle="--",
            label="Configured confidence threshold",
        )
        prediction_ax.legend()
    else:
        prediction_ax.text(0.5, 0.5, "No failure-risk outputs recorded", ha="center", va="center")
    prediction_ax.set_title("Simulated failure-risk confidence")
    prediction_ax.set_xlabel("Simulation time (hours)")
    prediction_ax.set_ylabel("Rule-based confidence (0 to 1)")
    prediction_ax.set_ylim(0, 1)
    prediction_ax.grid(True, alpha=0.3)

    benefits_ax = axes[1, 0]
    benefits = integration_stats["coordination_benefits"]
    categories = [name.replace("_", "\n").title() for name in benefits]
    values = list(benefits.values())
    bars = benefits_ax.bar(categories, values, color=["#e74c3c", "#f39c12", "#2ecc71"])
    benefits_ax.set_title("Illustrative coordination values")
    benefits_ax.set_ylabel("Modeled value (USD)")
    benefits_ax.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        offset = max(abs(value) * 0.02, 0.01)
        benefits_ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f"${value:,.0f}",
            ha="center",
            va="bottom",
        )

    wear_ax = axes[1, 1]
    motor_models = [
        model for model in simulator.digital_twin.physics_models
        if "_MOTOR" in model.component_id
    ][:3]
    for model in motor_models:
        data = simulator.physics_state_history.get(model.component_id, [])
        if data:
            times = [item["time"] for item in data]
            wear_levels = [item["wear_level"] for item in data]
            gate_id = model.component_id.split("_")[0][1:]
            wear_ax.plot(times, wear_levels, label=f"Gate {gate_id} motor")
    wear_ax.set_title("Illustrative motor wear state")
    wear_ax.set_xlabel("Simulation time (hours)")
    wear_ax.set_ylabel("Modeled wear level (dimensionless)")
    wear_ax.grid(True, alpha=0.3)
    if motor_models:
        wear_ax.legend()

    fig.tight_layout()
    plt.show()
    return fig


fig = create_digital_twin_visualizations(simulator, predictive_stats, integration_stats)

## Interpreting the results and comparing tiers

The numerical outputs describe the configured simulation run. In particular, the configured accuracy fields are not estimates of predictive performance: accuracy requires comparison with labeled outcomes, and this simulation supplies no independent ground truth. Likewise, the coordination-value calculation applies assumed dollar amounts to simulated events; it should not be interpreted as a realized financial benefit.

A qualitative comparison can still clarify the role of an integrated digital-twin architecture. The descriptions below are conceptual, not benchmark measurements. They do not establish that one tier is more reliable or cost-effective than another.

In [ ]:
def comprehensive_tier_comparison(
    simulator: DigitalTwinSimulator,
    integration_stats: Dict[str, Any],
) -> Dict[str, Dict[str, str]]:
    """Print a qualitative comparison without implying unmeasured scores."""
    comparison_data = {
        "Tier 1 (Optimal)": {
            "emphasis": "A defined optimization objective",
            "integration": "Limited in this conceptual comparison",
            "adaptability": "Depends on the specified model and inputs",
        },
        "Tier 2 (Heuristic)": {
            "emphasis": "Rule-based or heuristic decisions",
            "integration": "Can coordinate selected components",
            "adaptability": "Depends on the rules provided",
        },
        "Tier 3 (ACO)": {
            "emphasis": "Ant-colony optimization",
            "integration": "Depends on the modeled problem boundary",
            "adaptability": "Depends on the algorithm configuration",
        },
        "Tier 4 (DDQN)": {
            "emphasis": "Deep Q-network decision-making",
            "integration": "Depends on the state and action design",
            "adaptability": "Depends on training and deployment conditions",
        },
        "Tier 5 (Digital Twin)": {
            "emphasis": "Connected simulation and subsystem representation",
            "integration": "Can represent multiple interacting subsystems",
            "adaptability": "Depends on data, model validity, and maintenance",
        },
    }

    print("Qualitative comparison of solution tiers")
    print("These descriptions are not measured performance scores.\n")
    for tier, details in comparison_data.items():
        print(tier)
        for category, description in details.items():
            print(f"  {category.title()}: {description}")

    print("\nThis synthetic Tier 5 instance")
    print(f"  Configured gates: {simulator.digital_twin.num_gates}")
    print(f"  Configured sensors: {len(simulator.digital_twin.sensors)}")
    print(f"  Physics-model records: {len(simulator.digital_twin.physics_models)}")
    print(f"  Scenario definitions: {len(simulator.digital_twin.scenarios)}")
    print(f"  Simulated security events: {integration_stats['security_events']}")
    print(f"  Simulated energy events: {integration_stats['energy_events']}")
    print(
        "  Illustrative coordination value: "
        f"${integration_stats['total_coordination_value']:,.2f} (assumption-based)"
    )
    return comparison_data


final_comparison = comprehensive_tier_comparison(simulator, integration_stats)

## Why an integrated digital twin is useful

The defining idea of this tier is to represent related systems together. A gate event can affect traffic; an energy response can change gate processing; and component-condition estimates can inform maintenance planning. Connecting those representations makes it possible to inspect cross-system consequences within one model.

### Potential advantages

- **System integration:** one representation can connect gates, security, energy, and traffic.
- **Predictive workflows:** sensor histories and component-state estimates can be used to trigger alerts or maintenance reviews.
- **Scenario exploration:** a model can calculate what-if outcomes without directly changing live operations.
- **Operational visibility:** time-series records can help analysts examine how modeled conditions evolve.

### Challenges and limitations

- **Model validity:** simplified equations need calibration and independent validation before operational decisions rely on them.
- **Data integration:** real systems may use different formats, clocks, units, and quality standards.
- **Uncertainty:** sensor noise, missing data, parameter uncertainty, and unmodeled events can alter forecasts.
- **Maintenance and governance:** models, interfaces, access controls, and operational procedures require ongoing review.
- **Computational scale:** high-resolution models and large sensor networks require more resources than this small synthetic example.

### When this tier may be appropriate

An integrated digital-twin approach may be useful when a facility has interconnected subsystems, reliable data sources, a clear operational question, and the expertise to validate and maintain its models. The value depends on whether the model is fit for that question and whether its predictions are evaluated against real outcomes.

### Key takeaways

1. A digital twin is a connected model of a physical system; a simulation alone does not establish that the model is synchronized with, or validated against, a real facility.
2. The order of updates matters because later steps use state produced by earlier steps in the same simulation interval.
3. A rule-based confidence value is not a measured probability of failure. Predictive quality requires suitable validation data and evaluation methods.
4. What-if calculations are useful for tracing assumptions, but their outputs inherit those assumptions.
5. A credible operational deployment needs calibrated models, data-quality controls, uncertainty analysis, and monitoring of model performance.